# Breeze-ASR-Taigi on Google Colab

Evaluate [thc1006/breeze-asr-taigi](https://github.com/thc1006/breeze-asr-taigi) — a Taiwanese Hokkien ASR transcriber wrapping MediaTek's **Breeze-ASR-26**.

This notebook mirrors the project's Linux setup (`./install.sh`) so you can run it on a Colab GPU runtime.

## Before you run

Switch the runtime to a GPU: **Runtime → Change runtime type → Hardware accelerator → GPU** (a T4 is fine).

This notebook **forces the Faster-Whisper (CT2) engine** via `--engine fw` / `TAIGI_ASR_DEFAULT_ENGINE=fw`. A Colab T4 has ~16 GB VRAM, so the auto-router would otherwise pick the HuggingFace pipeline; we override that to evaluate the low-VRAM `int8_float16` path the project is actually tuned for.

Model download is ~2.9 GB on the first run.

## 1. Sanity-check the runtime

Confirms Python ≥ 3.10, `ffmpeg`, and that an NVIDIA GPU is visible.

In [ ]:
!python3 --version
!ffmpeg -version | head -n 1
!nvidia-smi

## 2. Clone the repository

In [ ]:
%cd /content
!git clone https://github.com/thc1006/breeze-asr-taigi.git
%cd /content/breeze-asr-taigi

## 3. Install dependencies

Colab already has a CUDA-enabled PyTorch installed, so we skip the explicit `cu121` torch reinstall that `install.sh` does (and avoid the long re-download). We install the project editable with the `hf` + `dev` extras, matching `pip install -e ".[hf,dev]"` from `install.sh`.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e ".[hf,dev]"

## 4. Force the Faster-Whisper engine

Set `TAIGI_ASR_DEFAULT_ENGINE=fw` for the rest of the notebook so anything that doesn't take an explicit `--engine` flag (e.g. the Gradio UI launcher) still uses Faster-Whisper instead of the auto-router's HuggingFace pick on a 16 GB T4.

In [ ]:
import os
os.environ["TAIGI_ASR_DEFAULT_ENGINE"] = "fw"
# Also export it for the !shell cells below.
%env TAIGI_ASR_DEFAULT_ENGINE=fw

## 5. Pre-download the Breeze-ASR-26 model (~2.9 GB)

Matches the final step of `install.sh`. Safe to skip — it will otherwise download on first transcription.

In [ ]:
!python -c "from taigi_asr.engines.faster_whisper import FasterWhisperEngine; FasterWhisperEngine.preload()"

## 6. Quick smoke test (Faster-Whisper)

Run the bundled sample (`data/test.m4a`, ~5.7 s) end-to-end on the `fw` engine and print the SRT.

In [ ]:
!taigi-asr data/test.m4a --engine fw --format srt --out /content/test.srt -v
!echo '--- /content/test.srt ---' && cat /content/test.srt

## 7. Transcribe your own audio

Upload a file with the Files panel (left sidebar) or run the cell below, then point `taigi-asr` at it.

Supported formats: `m4a / mp3 / wav / mp4 / mov / mkv / flac / ogg / webm` (anything ffmpeg accepts).

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    print('uploaded:', name)

In [ ]:
# Edit the filename below to match what you uploaded.
!taigi-asr /content/breeze-asr-taigi/your_audio.m4a --engine fw --format srt,txt,json -v

## 8. (Optional) Launch the Gradio UI with a public share link

`./start.sh` launches `python -m taigi_asr.ui.launcher`. On Colab the local 7860 port isn't reachable, so we add `--share` to get a Gradio tunnel URL. `TAIGI_ASR_DEFAULT_ENGINE=fw` (set in step 4) keeps the UI on Faster-Whisper.

This cell runs until you interrupt it.

In [ ]:
!TAIGI_ASR_DEFAULT_ENGINE=fw python -m taigi_asr.ui.launcher --share